# Практика · Self-attention покроково

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)

> ⏱ **Зошит навчає двадцять дві маленькі моделі** (сім у доборі швидкості й
> пʼятнадцять у порівнянні трьох способів на пʼятьох зернах). Моделі крихітні —
> чотири числа на вектор, — тож це секунди, а не хвилини. Заміряно
> `check_notebook.py` наодинці: **20 секунд** процесорного часу на чотирьох ядрах
> без відеокарти. Реального часу в двох прогонах вийшло 39 і 56 секунд — він
> залежить від того, чим ще зайнята машина, а процесорний не залежить.

## Задача, яку тут розвʼязано

Українське слово **«розділ»** в інтерфейсах програм означає щонайменше дві різні речі:

* у `elfutils`, `binutils`, `ld`, `gas` — **шматок обʼєктного файлу** (англійською *section*);
* у `parted`, `udisks2`, `anaconda` — **шматок диска** (англійською *partition*).

Пишеться однаково. У таблиці ембедингів це один рядок чисел — один на обидва значення,
бо таблиця має рівно стільки рядків, скільки слів у словнику.

**Наше завдання:** дати кожному входженню цього слова **власний** вектор, який знає,
про яке значення йдеться, і перевірити числом, що це справді вийшло.

Що зробимо:

1. зберемо з системи всі речення зі словом «розділ» у будь-якій формі й розмітимо їх за значенням;
2. **напишемо self-attention руками на numpy** й переконаємось, що він збігається з `nn.MultiheadAttention`;
3. навчимо модель із одним шаром уваги розрізняти значення — і порівняємо з двома базами на **пʼятьох зернах**;
4. розпишемо весь розрахунок на одному справжньому реченні, число за числом;
5. поміряємо, наскільки контекстні вектори одного слова розходяться між значеннями;
6. перевіримо дві властивості механізму: що буде без ділення на корінь і що буде з переставленими словами.

## 0 · Середовище

Грабля курсу: **кількість потоків фіксуємо до імпорту numpy**. Без цього
`time.process_time()` бреше в рази — потоки OpenMP крутяться в очікуванні, і це
очікування рахується як робота.

In [ ]:
import os
# ці чотири рядки мусять стояти ДО імпорту numpy і torch
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import sys, re, glob, gettext, math, random, time
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)          # те саме для torch, уже після імпорту
np.set_printoptions(precision=4, suppress=True, linewidth=140)

START = time.process_time()

print('python     ', sys.version.split()[0])
print('numpy      ', np.__version__)
print('torch      ', torch.__version__)
print('потоків    ', torch.get_num_threads())

## 1 · Звідки беремо дані

Корпус курсу — українські переклади інтерфейсів, які вже лежать у системі
(`/usr/share/locale/uk/LC_MESSAGES/*.mo`). Кожен файл — одна програма, і саме
**назва програми** дасть нам мітку значення: якщо рядок прийшов із `elfutils`,
то «розділ» там майже напевно про обʼєктний файл, а якщо з `parted` — про диск.

⚠️ Це **припущення, а не ручна розмітка**. Воно може помилятись у поодиноких рядках,
і чесніше сказати це відразу, ніж вдавати, що мітки бездоганні. Зате воно дає нам
півтори тисячі справжніх прикладів без жодної години розмітки.

Токенізатор — канонічний для курсу: апостроф усередині слова не розриває його.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"    # канон курсу, після lowercase
_token_re = re.compile(TOKEN_PATTERN)

def tokenize(text):
    """Ріже рядок на слова."""
    return _token_re.findall(text.lower())

# усі відмінкові форми слова: шукати будемо будь-яку з них
FORMS = {'розділ', 'розділу', 'розділи', 'розділів', 'розділом',
         'розділах', 'розділі', 'розділам', 'розділами'}

# програми, у яких «розділ» означає шматок обʼєктного файлу
SECTION = {'elfutils', 'bfd', 'binutils', 'gold', 'ld', 'gas', 'opcodes'}
# програми, у яких «розділ» означає шматок диска
PARTITION = {'parted', 'udisks2', 'anaconda', 'grub',
             'gnome-disk-utility', 'blivet', 'blivet-gui', 'gparted'}

def load_rows():
    """Читає каталоги перекладів і лишає лише речення зі словом «розділ»."""
    rows = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        program = path.split('/')[-1][:-3]
        if program not in SECTION and program not in PARTITION:
            continue
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                      # пошкоджений або нечитний файл просто пропускаємо
        label = 0 if program in SECTION else 1
        for src, dst in catalog._catalog.items():
            if isinstance(src, str) and isinstance(dst, str) \
               and len(dst) > 30 and 'Project-Id' not in dst:
                words = tokenize(dst)
                if 2 <= len(words) <= 30 and (FORMS & set(words)):
                    rows.append((words, label, program, dst))
    return rows

rows = load_rows()
print('речень зі словом «розділ»:', len(rows))
print('із них про обʼєктний файл :', sum(1 for r in rows if r[1] == 0))
print('із них про диск           :', sum(1 for r in rows if r[1] == 1))
print()
print('приклади «шматок обʼєктного файлу»:')
for words, label, program, raw in rows:
    if label == 0 and 4 <= len(words) <= 6:
        print('  %-10s %s' % (program, ' '.join(words)))
        if raw.count('%') > 5:
            break

Надрукуємо по кілька справжніх рядків кожного значення поруч — щоб побачити,
наскільки контекст різний, хоч слово те саме.

In [ ]:
def show(label, title, how_many=6):
    """Друкує кілька справжніх речень одного значення."""
    print(title)
    shown = 0
    for words, lab, program, raw in rows:
        if lab == label and 4 <= len(words) <= 7:
            print('  %-20s %s' % (program, ' '.join(words)))
            shown += 1
            if shown == how_many:
                break

show(0, 'РОЗДІЛ = шматок обʼєктного файлу')
print()
show(1, 'РОЗДІЛ = шматок диска')

## 2 · Поділ і словник

Речення в цих каталогах часто повторюються майже дослівно, тож спершу викидаємо
точні дублікати за набором слів — інакше та сама фраза потрапить і в навчальну, і
в перевірну частину, і перевірка стане брехливою.

Ділимо 70 / 15 / 15. Середня частина — **відкладена**: на ній ми добиратимемо
швидкість навчання. Підбирати гіперпараметр на перевірній вибірці означало б
підглядати у відповідь.

Словник будуємо **лише за навчальною частиною**.

In [ ]:
# 1. геть точні дублікати
seen, unique_rows = set(), []
for words, label, program, raw in rows:
    key = tuple(words)
    if key in seen:
        continue
    seen.add(key)
    unique_rows.append((words, label, program, raw))

# 2. перемішуємо з фіксованим зерном — поділ однаковий у всіх читачів
random.Random(0).shuffle(unique_rows)
n = len(unique_rows)
i_train, i_hold = int(n * 0.7), int(n * 0.85)
train_rows = unique_rows[:i_train]
hold_rows  = unique_rows[i_train:i_hold]
val_rows   = unique_rows[i_hold:]

# 3. словник із навчальної частини; слово, що трапилось один раз, не беремо
MIN_COUNT = 2
counts = Counter(w for words, _, _, _ in train_rows for w in words)
vocab = ['<unk>'] + sorted(w for w, c in counts.items() if c >= MIN_COUNT)
word_to_id = {w: i for i, w in enumerate(vocab)}

def encode(part):
    """Перетворює речення на номери слів і запамʼятовує, де стоїть слово «розділ»."""
    out = []
    for words, label, program, raw in part:
        position = [i for i, w in enumerate(words) if w in FORMS][0]
        ids = [word_to_id.get(w, 0) for w in words]
        out.append((ids, label, position, words, program, raw))
    return out

train, hold, val = encode(train_rows), encode(hold_rows), encode(val_rows)

print('після зняття дублікатів:', n, 'речень')
print('навчальних  ', len(train))
print('відкладених ', len(hold))
print('перевірних  ', len(val))
print('словник     ', len(vocab))
print('класи в навчальній :', dict(Counter(y for _, y, _, _, _, _ in train)))
print('класи в перевірній :', dict(Counter(y for _, y, _, _, _, _ in val)))

Класи нерівні приблизно як чотири до одного. Через це **звичайна частка
правильних відповідей нам не годиться**: модель, яка завжди відповідає «обʼєктний
файл», дістала б 0.80, нічого не навчившись.

Беремо **збалансовану влучність** — середнє з двох повнот. Такій ледачій моделі
вона дає рівно 0.5000, і це наша база.

In [ ]:
def balanced_accuracy(model, data, batch_size=256):
    """Середнє з повноти по кожному класу. База «завжди більшість» дає 0.5000."""
    model.eval()
    correct, total = [0, 0], [0, 0]
    with torch.no_grad():
        for ids, labels, positions in make_batches(data, batch_size, seed=0):
            predicted = model(ids, positions).argmax(dim=1)
            for c in (0, 1):
                chosen = (labels == c)
                correct[c] += (predicted[chosen] == c).sum().item()
                total[c] += int(chosen.sum())
    return 0.5 * (correct[0] / max(total[0], 1) + correct[1] / max(total[1], 1))

def make_batches(data, batch_size, seed):
    """Складає батчі з речень ОДНАКОВОЇ довжини.

    Так батч не потребує доповнення порожніми токенами — а отже, і маскування,
    яке в цій темі ми свідомо не використовуємо.
    """
    by_length = defaultdict(list)
    for ids, label, position, _, _, _ in data:
        by_length[len(ids)].append((ids, label, position))
    batches = []
    for length, items in by_length.items():
        for start in range(0, len(items), batch_size):
            chunk = items[start:start + batch_size]
            batches.append((torch.tensor([c[0] for c in chunk]),
                            torch.tensor([c[1] for c in chunk]),
                            torch.tensor([c[2] for c in chunk])))
    random.Random(seed).shuffle(batches)
    return batches

lengths = Counter(len(ids) for ids, _, _, _, _, _ in train)
print('різних довжин речень у навчальній частині:', len(lengths))
print('найчастіші довжини:', lengths.most_common(5))
print('батчів за одну епоху при розмірі 32:', len(make_batches(train, 32, 0)))

## 3 · Self-attention, написаний руками

Тепер найголовніше. Уся операція — пʼять рядків на numpy, і жодної магії в них немає.

Нагадаю формулу, яку ми реалізуємо:

```
Attention(Q, K, V) = softmax( Q·Kᵀ / √d ) · V
```

* `Q = X·W_Q` — **запити**: чого шукає кожна позиція;
* `K = X·W_K` — **ключі**: чим кожна позиція себе рекламує;
* `V = X·W_V` — **значення**: що кожна позиція віддає тому, хто її вибрав;
* `Q·Kᵀ` — таблиця оцінок `n×n`: наскільки запит `i` підходить до ключа `j`;
* ділення на `√d` тримає розкид оцінок сталим незалежно від довжини вектора;
* `softmax` по кожному рядку окремо перетворює оцінки на пропорції;
* множення на `V` — та сама зважена сума.

Softmax пишемо через віднімання максимуму рядка. Це не оптимізація, а
**захист від переповнення**: `exp(1000)` дає нескінченність, і весь рядок
перетворюється на `nan`. Віднімання сталої результат не змінює — вона скорочується
в чисельнику й знаменнику.

In [ ]:
def softmax_rows(scores):
    """Softmax по кожному рядку окремо. Відняли максимум, щоб exp не переповнився."""
    shifted = scores - scores.max(axis=-1, keepdims=True)
    weights = np.exp(shifted)
    return weights / weights.sum(axis=-1, keepdims=True)

def self_attention(X, W_q, W_k, W_v):
    """Один шар self-attention з однією головою. Повертає вихід і таблицю ваг."""
    d = X.shape[1]
    Q = X @ W_q                       # чого шукає кожна позиція
    K = X @ W_k                       # чим кожна позиція себе рекламує
    V = X @ W_v                       # що кожна позиція віддає
    scores = Q @ K.T / math.sqrt(d)   # ділимо, щоб softmax не насичувався
    A = softmax_rows(scores)          # рядок ваг для кожної позиції
    return A @ V, A                   # зважена сума значень

# найдешевша перевірка, яка ловить помилку в осі softmax:
# рядки таблиці ваг МУСЯТЬ сумуватись в одиницю, хай там які ваги
rng = np.random.default_rng(0)
X_test = rng.normal(size=(5, 4)).astype(np.float32)
_, A_test = self_attention(X_test, rng.normal(size=(4, 4)).astype(np.float32),
                           rng.normal(size=(4, 4)).astype(np.float32),
                           rng.normal(size=(4, 4)).astype(np.float32))
print('суми рядків таблиці ваг:', A_test.sum(axis=1))
assert np.allclose(A_test.sum(axis=1), 1.0), 'softmax узято не по тій осі!'
print('✅ кожен рядок сумується в одиницю')

## 4 · Звірка з бібліотечною реалізацією

А тепер перевіримо себе тим самим, чим перевіряють у справжній роботі: порівняємо
з `torch.nn.MultiheadAttention`.

Щоб порівняння було чесним, треба **віддати бібліотеці рівно наші ваги**. У torch
три матриці лежать склеєні в одному тензорі `in_proj_weight` — спершу `W_q`, потім
`W_k`, потім `W_v`, кожна в **транспонованому** вигляді (бо `nn.Linear` рахує
`x @ W.T`). Після уваги torch додатково множить результат на четверту матрицю
`out_proj`; у нас такої немає, тож ставимо туди одиничну — множення на неї нічого
не міняє.

⚠️ **Питати «чи результати однакові» — неправильно.** Додавання чисел із рухомою
комою не є асоціативним: `(a+b)+c` і `a+(b+c)` можуть дати різні останні розряди.
Бібліотека складає в тому порядку, який швидший для процесора, ми — зліва направо.
Правильне питання: **чи різниця не більша за одиницю останнього розряду**.

In [ ]:
def compare_with_torch(d_model, seed, n_positions=5):
    """Наша реалізація проти nn.MultiheadAttention на однакових вагах."""
    torch.manual_seed(seed)
    mha = nn.MultiheadAttention(d_model, num_heads=1, bias=False, batch_first=True)
    x = torch.randn(1, n_positions, d_model)

    with torch.no_grad():
        # три матриці torch тримає склеєними в одному тензорі, у транспонованому вигляді
        w_q, w_k, w_v = mha.in_proj_weight.chunk(3, dim=0)
        w_out = mha.out_proj.weight

        ours, _ = self_attention(x[0].numpy(),
                                 w_q.numpy().T, w_k.numpy().T, w_v.numpy().T)
        ours = ours @ w_out.numpy().T          # четверта матриця, яку torch множить після
        theirs, _ = mha(x, x, x, need_weights=False)
        theirs = theirs[0].numpy()

    return np.abs(ours - theirs).max(), np.array_equal(ours, theirs)

eps32 = np.finfo(np.float32).eps
print('машинний епсилон float32: %.3e' % eps32)
print()
print('зерно | найбільша різниця | у епсилонах | побітово однакові')
for seed in (0, 1, 2, 3, 4):
    diff, identical = compare_with_torch(16, seed)
    print('%5d | %17.3e | %11.2f | %s' % (seed, diff, diff / eps32, identical))
    assert diff < 10 * eps32, 'розрахунок розійшовся більше, ніж на округлення!'
print()
print('✅ збігається в межах одиниці останнього розряду')
print('   і НЕ збігається побітово — так і має бути')

Різниця тримається на рівні одного машинного епсилона, тобто наші числа й
бібліотечні — **сусіди на числовій осі**, між ними немає нічого. А побітово вони не
збіглися **жодного разу**, і це нормальний, очікуваний результат, а не дефект.

Мораль на все життя: у перевірках пишіть не `==`, а порівняння з допуском, і допуск
беріть від епсилона того типу, у якому рахуєте.

## 5 · Модель

Далі — три моделі, які відрізняються рівно одним: **звідки береться вектор, за яким
ми судимо про значення слова**.

| модель | вектор для класифікації |
|---|---|
| `AttentionModel` | вихід self-attention **на позиції слова «розділ»** |
| `StaticModel` | статичний ембединг того самого слова, без жодного контексту |
| `BagModel` | середнє ембедингів усього речення — класичний мішок слів |

`StaticModel` — це база, яка показує, скільки можна витиснути **без контексту
взагалі**. `BagModel` — база, яка контекст бачить, але не вміє віддати його
конкретній позиції.

In [ ]:
class AttentionModel(nn.Module):
    """Ембединги -> один шар self-attention -> вектор на позиції слова -> клас."""
    def __init__(self, vocab_size, d_model, n_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.head = nn.Linear(d_model, n_classes)
        self.d_model = d_model

    def attend(self, x):
        q, k, v = self.w_q(x), self.w_k(x), self.w_v(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_model)
        weights = torch.softmax(scores, dim=-1)
        return weights @ v, weights

    def forward(self, ids, positions):
        contextual, _ = self.attend(self.embedding(ids))
        # беремо вектор рівно з тієї позиції, де стоїть слово «розділ»
        chosen = contextual[torch.arange(len(positions)), positions]
        return self.head(chosen)

class StaticModel(nn.Module):
    """Без уваги взагалі: судимо за статичним ембедингом самого слова."""
    def __init__(self, vocab_size, d_model, n_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, ids, positions):
        chosen = self.embedding(ids)[torch.arange(len(positions)), positions]
        return self.head(chosen)

class BagModel(nn.Module):
    """Мішок слів: середнє ембедингів усього речення."""
    def __init__(self, vocab_size, d_model, n_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, ids, positions):
        return self.head(self.embedding(ids).mean(dim=1))

D_MODEL = 4      # смішно мало для справжньої моделі, зате все влазить на екран
EPOCHS = 15
SEEDS = (0, 1, 2, 3, 4)   # пʼять прогонів: три дають надто вузьку й оманливу купу

def train_model(model, data, epochs, learning_rate, seed, batch_size=32):
    """Навчання. Помилки на рідкісному класі важать більше, бо класи нерівні."""
    class_counts = Counter(y for _, y, _, _, _, _ in data)
    weights = torch.tensor([len(data) / (2 * class_counts[c]) for c in (0, 1)],
                           dtype=torch.float32)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    for epoch in range(epochs):
        model.train()
        for ids, labels, positions in make_batches(data, batch_size, seed * 100 + epoch):
            optimizer.zero_grad()
            loss_fn(model(ids, positions), labels).backward()
            optimizer.step()
    return model

print('параметрів у шарі уваги при d = %d: 3 × %d × %d = %d'
      % (D_MODEL, D_MODEL, D_MODEL, 3 * D_MODEL * D_MODEL))
print('це і є ВЕСЬ набір ваг self-attention — від довжини речення він не залежить')

## 6 · Швидкість навчання добираємо на відкладеній вибірці

Гіперпараметр не можна добирати на перевірній частині: це підглядання у відповідь.
Тому в нас є окрема відкладена вибірка, і сітка йде від дуже малого кроку до дуже
великого — доки мінімум не опиниться **всередині** діапазону, а не на краю. Якщо
найкраще значення на краю, сітка закоротка і її треба продовжити.

In [ ]:
t0 = time.process_time()
LR_GRID = [0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
grid_scores = []

for learning_rate in LR_GRID:
    torch.manual_seed(0)
    model = train_model(AttentionModel(len(vocab), D_MODEL), train,
                        EPOCHS, learning_rate, seed=0)
    grid_scores.append(balanced_accuracy(model, hold))
    print('  крок %-6g відкладена %.4f' % (learning_rate, grid_scores[-1]))

best_index = int(np.argmax(grid_scores))
LEARNING_RATE = LR_GRID[best_index]
print()
print('обрано крок навчання:', LEARNING_RATE)
print('мінімум усередині сітки:', 0 < best_index < len(LR_GRID) - 1)
print('процесорних секунд: %.1f' % (time.process_time() - t0))

## 7 · Навчання: три способи, три зерна

Одне зерно нічого не доводить: різниця, менша за розкид по зернах, не є різницею.
Тому кожну модель навчаємо **пʼять разів** і дивимось не тільки на середнє, а й на
купу окремих значень.

Чому саме пʼять, а не три. Вузька купа з трьох прогонів оманлива: третій прогін може
випадково лягти поруч із двома першими, і різниця здасться доведеною там, де її
немає. Дешевше додати два зерна, ніж повірити у власний шум.

In [ ]:
t0 = time.process_time()
results = {}

for name, cls in [('self-attention', AttentionModel),
                  ('статичний ембединг', StaticModel),
                  ('мішок слів', BagModel)]:
    scores, models = [], []
    for seed in SEEDS:
        torch.manual_seed(seed)
        model = train_model(cls(len(vocab), D_MODEL), train, EPOCHS, LEARNING_RATE, seed)
        scores.append(balanced_accuracy(model, val))
        models.append(model)
    results[name] = (np.array(scores), models)
    print('%-20s %s | середнє %.4f ±%.4f | купа %.4f…%.4f'
          % (name, ' '.join('%.4f' % s for s in scores),
             np.mean(scores), np.std(scores, ddof=1), min(scores), max(scores)))

print()
print('%-20s %.4f  (завжди відповідати найчастішим класом)' % ('база', 0.5))
print('процесорних секунд: %.1f' % (time.process_time() - t0))

Читаємо результат чесно.

* **Проти статичного ембединга self-attention виграє впевнено**: купи по пʼятьох
  зернах не перетинаються — найгірший прогін з увагою кращий за найкращий без неї.
  Це і є головний результат зошита: контекст справді потрапив у вектор слова.
* **Проти мішка слів різниці немає**: купи перетинаються. Мішок слів бачить ті самі
  слова речення, і на такій маленькій задачі цього досить. Різниця не в точності, а
  в тому, **що ми отримуємо на виході**: мішок слів дає один вектор на все речення,
  а self-attention — свій вектор кожній позиції. Наступні клітинки саме це й
  використовують.

Далі розбиратимемо механізм на **одному** з пʼятьох прогонів. Який саме взяти —
питання не принципове: механіка в усіх однакова, різні лише числа. Беремо зерно 2 і
одразу друкуємо його влучність, щоб було видно, що це не найкращий прогін із пʼятьох.

In [ ]:
attention_scores, attention_models = results['self-attention']
SHOW_SEED = 2
model = attention_models[SEEDS.index(SHOW_SEED)]
print('розбираємо зерно %d, збалансована влучність %.4f'
      % (SHOW_SEED, attention_scores[SEEDS.index(SHOW_SEED)]))
print('найкраще з пʼятьох зерен: %.4f' % attention_scores.max())

# витягуємо матриці у звичному вигляді: рядок вектора множиться на матрицю справа
with torch.no_grad():
    W_q = model.w_q.weight.T.numpy().copy()
    W_k = model.w_k.weight.T.numpy().copy()
    W_v = model.w_v.weight.T.numpy().copy()
print('розмір кожної матриці:', W_q.shape)
print('W_q:'); print(W_q)

## 8 · Увесь розрахунок на одному справжньому реченні

Візьмімо реальне речення з корпусу — воно трапляється в `elfutils` шість разів:

```
не вдалося отримати заголовок розділу
```

Пʼять слів, отже `n = 5`; вектор кожного слова має чотири числа, отже `d = 4`.
Пройдемо всі шість кроків і надрукуємо кожен проміжний результат.

In [ ]:
SENTENCE = ['не', 'вдалося', 'отримати', 'заголовок', 'розділу']
assert all(w in word_to_id for w in SENTENCE), 'слово поза словником — числа були б не ті'

with torch.no_grad():
    X = model.embedding(torch.tensor([[word_to_id[w] for w in SENTENCE]]))[0].numpy()

print('речення:', ' '.join(SENTENCE))
print()
print('крок 0 · X — вхід шару, рядок на слово:')
for word, row in zip(SENTENCE, X):
    print('  %-12s %s' % (word, row))

**Крок 1.** Три проєкції: множимо `X` на три навчені матриці й дістаємо запити,
ключі та значення.

In [ ]:
Q = X @ W_q
K = X @ W_k
V = X @ W_v

for name, matrix, role in [('Q', Q, 'запити: чого шукає кожна позиція'),
                           ('K', K, 'ключі: чим кожна позиція себе рекламує'),
                           ('V', V, 'значення: що кожна позиція віддає')]:
    print('%s — %s' % (name, role))
    for word, row in zip(SENTENCE, matrix):
        print('  %-12s %s' % (word, row))
    print()

**Крок 2.** Таблиця оцінок: кожен запит скалярно множимо на кожен ключ.
Розпишемо одну клітинку вручну, щоб було видно, що це просто чотири множення
й три додавання.

In [ ]:
i, j = 0, 1                                   # запит слова «не», ключ слова «вдалося»
products = Q[i] * K[j]                        # почленний добуток
print('q(%s) = %s' % (SENTENCE[i], Q[i]))
print('k(%s) = %s' % (SENTENCE[j], K[j]))
print('почленні добутки :', products)
print('їхня сума        : %.4f' % products.sum())

scores = Q @ K.T
print('та сама клітинка з матриці: %.4f' % scores[i, j])
assert abs(products.sum() - scores[i, j]) < 1e-5, 'ручний рахунок розійшовся з матричним'
print()
print('уся таблиця оцінок S = Q·Kᵀ (рядок — хто питає, стовпець — кого):')
print('%-12s %s' % ('', ' '.join('%9s' % w[:9] for w in SENTENCE)))
for word, row in zip(SENTENCE, scores):
    print('%-12s %s' % (word, ' '.join('%9.4f' % v for v in row)))

**Крок 3.** Ділимо всю таблицю на `√d`. У нас `d = 4`, отже дільник рівно **2**.

**Крок 4.** Softmax по кожному рядку окремо — з чисел виходять пропорції.

In [ ]:
d = X.shape[1]
scaled = scores / math.sqrt(d)
A = softmax_rows(scaled)

print('оцінки, поділені на √%d = %.0f:' % (d, math.sqrt(d)))
for word, row in zip(SENTENCE, scaled):
    print('  %-12s %s' % (word, ' '.join('%8.4f' % v for v in row)))
print()
print('ваги уваги A = softmax(S / √d):')
print('%-12s %s' % ('', ' '.join('%9s' % w[:9] for w in SENTENCE)))
for word, row in zip(SENTENCE, A):
    print('%-12s %s' % (word, ' '.join('%9.4f' % v for v in row)))
print()
print('суми рядків:', A.sum(axis=1))
print('увага до себе (діагональ):', A.diagonal().round(4),
      '| середнє %.4f при рівномірних %.4f' % (A.diagonal().mean(), 1 / len(SENTENCE)))
print()
# три числа, якими зручно описати карту уваги одним рядком
print('|A − Aᵀ| максимум     : %.4f   (нуль означав би симетричну карту)'
      % np.abs(A - A.T).max())
row_entropy = -(A * np.log(A)).sum(axis=1)
print('ентропія рядків, нат  :', row_entropy.round(4),
      '| рівномірна ln 5 = %.4f' % math.log(len(SENTENCE)))
print('розкид самих оцінок S : %.4f' % scores.std())

**Крок 5.** Зважена сума: множимо таблицю ваг на матрицю значень. Знову
розпишемо одне число руками.

In [ ]:
Y = A @ V

position, component = 0, 0                     # вихід слова «не», перше з чотирьох чисел
contributions = A[position] * V[:, component]
print('внески у вихід «%s», число %d:' % (SENTENCE[position], component + 1))
for word, weight, value, contribution in zip(SENTENCE, A[position],
                                             V[:, component], contributions):
    print('  %-12s вага %.4f × значення %8.4f = %8.4f' % (word, weight, value, contribution))
print('  сума                                     = %8.4f' % contributions.sum())
print('  у матриці Y стоїть                        %8.4f' % Y[position, component])
print()
print('Y — вихід шару, рядок на слово:')
for word, row in zip(SENTENCE, Y):
    print('  %-12s %s' % (word, row))

## 9 · Той самий розрахунок бібліотекою

Тепер віддамо `nn.MultiheadAttention` наші **навчені** ваги й перевіримо, що вона
дає те саме — і ваги уваги, і вихід.

In [ ]:
library_attention = nn.MultiheadAttention(D_MODEL, num_heads=1, bias=False, batch_first=True)
with torch.no_grad():
    # torch тримає три матриці склеєними, у транспонованому вигляді
    library_attention.in_proj_weight.copy_(
        torch.cat([model.w_q.weight, model.w_k.weight, model.w_v.weight], dim=0))
    library_attention.out_proj.weight.copy_(torch.eye(D_MODEL))   # у нас четвертої матриці немає
    if library_attention.out_proj.bias is not None:
        library_attention.out_proj.bias.zero_()
    x_torch = torch.tensor(X).unsqueeze(0)
    Y_library, A_library = library_attention(x_torch, x_torch, x_torch)

diff_output = np.abs(Y - Y_library[0].numpy()).max()
diff_weights = np.abs(A - A_library[0].numpy()).max()
print('найбільша різниця у вихідних векторах: %.3e  (%.2f епсилона)'
      % (diff_output, diff_output / eps32))
print('найбільша різниця у вагах уваги      : %.3e  (%.2f епсилона)'
      % (diff_weights, diff_weights / eps32))
print('побітово однакові:', np.array_equal(Y, Y_library[0].numpy()))
assert np.allclose(Y, Y_library[0].numpy(), atol=1e-5), 'розрахунок розійшовся!'
print()
print('✅ збігається')

## 10 · Головна перевірка: контекст справді потрапив у вектор

Тепер те, заради чого все й затівалось. Візьмімо друге справжнє речення з корпусу,
у якому те саме слово означає **інше**:

```
речення 1  не вдалося отримати заголовок розділу     (розділ обʼєктного файлу)
речення 2  розмір розділу який слід створити         (розділ диска)
```

Проженімо обидва крізь той самий шар із тими самими вагами й порівняймо вектор
слова «розділу» до й після.

In [ ]:
SENTENCE_2 = ['розмір', 'розділу', 'який', 'слід', 'створити']
assert all(w in word_to_id for w in SENTENCE_2)

with torch.no_grad():
    X2 = model.embedding(torch.tensor([[word_to_id[w] for w in SENTENCE_2]]))[0].numpy()
Y2, A2 = self_attention(X2, W_q, W_k, W_v)

def cosine(a, b):
    """Косинус між векторами: 1 — той самий напрямок, −1 — протилежний."""
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

p1 = SENTENCE.index('розділу')
p2 = SENTENCE_2.index('розділу')

print('рядок уваги слова «розділу»')
print('  у реченні 1:', dict(zip(SENTENCE, A[p1].round(4))))
print('  у реченні 2:', dict(zip(SENTENCE_2, A2[p2].round(4))))
print()
print('вектор слова «розділу» ДО шару (статичний ембединг):')
print('  речення 1:', X[p1])
print('  речення 2:', X2[p2])
print('  косинус  : %.4f' % cosine(X[p1], X2[p2]))
print()
print('вектор слова «розділу» ПІСЛЯ шару (контекстний):')
print('  речення 1:', Y[p1])
print('  речення 2:', Y2[p2])
print('  косинус  : %.4f' % cosine(Y[p1], Y2[p2]))

Одиниця вгорі — не результат, а тавтологія: статичний ембединг береться з
таблиці за номером слова, слово те саме, отже вектор той самий. У цьому й полягала
задача.

Число внизу — уже результат.

Два речення — це два речення. Поміряймо те саме на **всій перевірній вибірці**:
чи справді контекстні вектори одного слова групуються за значенням.

In [ ]:
def contextual_vectors(model, data):
    """Для кожного речення повертає контекстний вектор слова «розділ» і мітку."""
    vectors, labels = [], []
    with torch.no_grad():
        for ids, label, position, _, _, _ in data:
            hidden, _ = model.attend(model.embedding(torch.tensor([ids])))
            vectors.append(hidden[0, position].numpy())
            labels.append(label)
    return np.array(vectors), np.array(labels)

vectors, labels = contextual_vectors(model, val)
normalized = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
cosines = normalized @ normalized.T

same_sense, other_sense = [], []
for a in range(len(labels)):
    for b in range(a + 1, len(labels)):
        (same_sense if labels[a] == labels[b] else other_sense).append(cosines[a, b])

print('контекстних векторів слова «розділ» у перевірній вибірці:', len(vectors))
print('середній косинус у межах одного значення : %.4f' % np.mean(same_sense))
print('середній косинус між різними значеннями  : %.4f' % np.mean(other_sense))
print('розрив                                   : %.4f'
      % (np.mean(same_sense) - np.mean(other_sense)))
print()
print('для статичних ембедингів однієї словоформи обидва числа дорівнювали б 1.0000,')
print('а розрив був би нульовим — саме тому вони й не розрізняють значень')

## 11 · Дві властивості механізму, які варто побачити руками

**Перша: що станеться, якщо не поділити на `√d`.**

Softmax перетворює числа на ваги так, що найбільше забирає майже все. Що більший
розкид оцінок, то жорсткіше це «майже все» — і то менше ваги здатні зрушити, коли
оцінки трохи змінюються. Останнє міряємо сумою `p·(1−p)` по рядку: якщо одна вага
дорівнює одиниці, а решта нулі, ця сума нульова, і навчання в цьому місці стоїть.

In [ ]:
def sharpness(scores, divisor):
    """Наскільки гострі ваги при заданому дільнику й скільки в них лишилось руху."""
    weights = softmax_rows(scores / divisor)
    max_weight = weights.max(axis=1).mean()
    movement = (weights * (1 - weights)).sum(axis=1).mean()
    return max_weight, movement

print('дільник        макс. вага   сила градієнта')
for name, divisor in [('без ділення', 1.0),
                      ('÷ √d = 2', math.sqrt(d)),
                      ('÷ d = 4', float(d)),
                      ('оцінки роздуто ×4', 0.25)]:
    max_weight, movement = sharpness(scores, divisor)
    print('%-18s %.4f       %.4f' % (name, max_weight, movement))
print()
print('стеля сили градієнта на пʼятьох ключах (рівномірні ваги): %.4f' % (5 * 0.2 * 0.8))
print('розкид самих оцінок: від %.4f до %.4f' % (scores.min(), scores.max()))

На нашій маленькій моделі оцінки й так невеликі, тож ефект помірний. Але
подивіться на останній рядок: варто штучно роздути оцінки вчетверо — і сила
градієнта помітно падає. У справжній моделі з `d = 512` оцінки роздуваються самі
собою, бо доданків у скалярному добутку в сто двадцять вісім разів більше.

**Друга: порядок слів шар не бачить взагалі.**

У формулі немає жодного місця, де згадувалась би позиція. Перевіримо: переставимо
слова, проженемо крізь шар і розставимо результат назад.

In [ ]:
permutation = [3, 0, 4, 1, 2]                       # довільна перестановка позицій
X_shuffled = X[permutation]
Y_shuffled, _ = self_attention(X_shuffled, W_q, W_k, W_v)

# розставляємо рядки назад у початковий порядок
Y_restored = np.empty_like(Y_shuffled)
for new_position, old_position in enumerate(permutation):
    Y_restored[old_position] = Y_shuffled[new_position]

print('подали в порядку:', [SENTENCE[i] for i in permutation])
print('найбільша розбіжність із початковим виходом: %.3e'
      % np.abs(Y_restored - Y).max())
print('у машинних епсилонах float32: %.2f' % (np.abs(Y_restored - Y).max() / eps32))
assert np.allclose(Y_restored, Y, atol=1e-5)
print()
print('✅ вихід той самий, лише переставлений разом із входом')
print('   для шару «розділ диска» і «диска розділ» — одне й те саме')

Це називають **еквіваріантністю щодо перестановки**. Лікується не переробкою
self-attention, а додаванням відомостей про позицію до векторів ще до входу в шар —
але це вже інша тема.

## 12 · Що вийшло і де межа

Підсумок зошита в трьох числах.

In [ ]:
print('ЗАДАЧА · розрізнити два значення слова «розділ» за контекстом')
print('  перевірних прикладів:', len(val))
print('  база «завжди найчастіший клас»: 0.5000')
print()
print('РЕЗУЛЬТАТ · збалансована влучність, пʼять зерен')
for name in ('self-attention', 'статичний ембединг', 'мішок слів'):
    s = results[name][0]
    print('  %-20s %.4f ±%.4f   купа: %.4f … %.4f'
          % (name, s.mean(), s.std(ddof=1), s.min(), s.max()))
print()
print('ГОЛОВНЕ · вектор слова «розділу» у двох реченнях, косинус')
print('  до шару  : %.4f' % cosine(X[p1], X2[p2]))
print('  після    : %.4f' % cosine(Y[p1], Y2[p2]))
print()
print('процесорних секунд на весь зошит: %.1f' % (time.process_time() - START))

**Де межа цього зошита — чесно.**

1. **Мітки не розмічені людиною.** Ми вивели значення слова з назви програми. У
   поодиноких рядках це напевно неправда: у `grub` трапляється й «розділ довідки».
2. **Даних мало** — близько двохсот перевірних прикладів. Один приклад важить
   пів відсотка, і саме тому ми скрізь беремо пʼять зерен і дивимось на купу, а не
   на середнє.
3. **`d = 4` — іграшковий розмір.** Він обраний, щоб усе влазило на екран і
   рахувалось руками. Справжні моделі беруть 128–768, і там числа інші.
4. **Мішок слів не програв.** На такій маленькій задачі він дає те саме. Виграш
   self-attention тут не в точності, а в тому, що на виході — вектор **кожної
   позиції**, а не один вектор на речення.
5. **Одна голова, без маскування, без позицій.** Це навмисне спрощення: так видно
   сам механізм.

---

## Завдання

### 🟢 Рівень 1 — База

Візьміть інше багатозначне слово нашого корпусу — наприклад **«ключ»** (у `gnupg2`
це криптографічний ключ, у `postgres-15` — ключ таблиці, у `NetworkManager` — ключ
мережі) — і повторіть увесь шлях: зберіть речення, розмітьте за програмами, навчіть
модель, поміряйте косинуси.

**Зроблено, якщо:** ви надрукували середній косинус у межах одного значення й між
різними значеннями і можете сказати, чи розрив більший за розкид по пʼятьох зернах.

### 🟡 Рівень 2 — Плюс

Поміняйте `D_MODEL` з 4 на 8, 16 і 32, залишивши все інше. Для кожного розміру
надрукуйте збалансовану влучність (пʼять зерен) і середню максимальну вагу в рядках
уваги.

**Зроблено, якщо:** ви побудували таблицю «розмір вектора → влучність → гострота
уваги» і сказали словами, чи допомагає більший вектор на цій задачі й чому.

### 🔴 Рівень 3 — Виклик

Приберіть ділення на `√d` із `self_attention` і навчіть модель заново, пʼять зерен,
на всіх чотирьох розмірах із рівня 2. Потім поверніть ділення й повторіть.

**Зроблено, якщо:** ви показали числом, на якому розмірі вектора відсутність
ділення починає шкодити, і підкріпили це виміряною силою градієнта `Σ p(1−p)`
у навчених моделях обох варіантів.

### Підказки

* Форми слова зручно зібрати простим списком — морфологічний аналізатор тут не
  потрібен, форм небагато.
* Якщо для нового слова замало даних, послабте `MIN_COUNT` до 1: словник виросте,
  але рідкісні слова хоч якось навчаться.
* Сила градієнта рахується одним рядком: `(A * (1 - A)).sum(axis=1).mean()`.